In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from keras import initializers
import ast

Datasets = []
PREDICTORS = ["PwmD", "PwmE", "Theta"]   
TARGET_INT = ["Theta", "X", "Y"]  
TARGET = ["DeltaTheta", "DeltaX", "DeltaY",]
SETS = [
    "Train_1",
    "Train_2",
    "Test_1",
    "Test_2",
    "Test_3",
    "Val",
    "Lsg_1",
    "Lsg_2",
]
     
TIME_STEPS = 3
TS = 0.07

In [2]:
for i in range(4):
    Dataset = pd.read_excel(f"./../../RotedData/Data.xlsx", f"D{i+1}")   
    Datasets.append(Dataset)

for i in range(4):   
    Dataset = pd.read_csv(f"./../../Data/Data{i + 1}.csv")  
    Datasets.append(Dataset)
    
    
for i in range(len(Datasets)):
    Dataset = Datasets[i].copy()

    for var in TARGET_INT:
        Dataset[f"Delta{var}"] = (Dataset[var].shift(-1) - Dataset[var]) / TS

    Dataset = Dataset.dropna(subset=[f"Delta{var}" for var in TARGET_INT])

    Datasets[i] = Dataset

In [3]:
results = [] 
for target in TARGET_INT:
    result = pd.read_excel(f"./models/{target}.xlsx")
    results.append(result)

In [4]:
NormDatasets = []

INPUT_SCALER = StandardScaler()

OUT_SCALERS = {
    "DeltaTheta": StandardScaler(),
    "DeltaX": StandardScaler(),
    "DeltaY": StandardScaler()
}

# ======================
# Train 1
# ======================
Train1 = Datasets[0].copy()

Train1[PREDICTORS] = INPUT_SCALER.fit_transform(Train1[PREDICTORS])

for t in TARGET:
    Train1[[t]] = OUT_SCALERS[t].fit_transform(Train1[[t]])

NormDatasets.append(Train1)

# ======================
# Train 2
# ======================
Train2 = Datasets[1].copy()

Train2[PREDICTORS] = INPUT_SCALER.transform(Train2[PREDICTORS])

for t in TARGET:
    Train2[[t]] = OUT_SCALERS[t].transform(Train2[[t]])

NormDatasets.append(Train2)

# concatena treino
Train = pd.concat([Train1, Train2], ignore_index=True)

# ======================
# Testes + Val
# ======================
for i in range(6):

    CurrentTestDataset = Datasets[i + 2].copy()

    CurrentTestDataset[PREDICTORS] = INPUT_SCALER.transform(
        CurrentTestDataset[PREDICTORS]
    )

    for t in TARGET:
        CurrentTestDataset[[t]] = OUT_SCALERS[t].transform(
            CurrentTestDataset[[t]]
        )

    NormDatasets.append(CurrentTestDataset)

In [7]:
import pandas as pd

def PickModels(df, target, tr1=0.0, tr2=0.0, v1=0.0, t3=0.0):

    r2_tr1 = f"R2_Train_1_{target}"
    r2_tr2 = f"R2_Train_2_{target}"
    r2_val = f"R2_Val_{target}"
    r2_t3 = f"R2_Test_3_{target}"

    # filtro mínimo
    filtered = df[
        (df[r2_tr1] > tr1) &
        (df[r2_tr2] > tr2) &
        (df[r2_t3] > t3) &
        (df[r2_val] > v1)
    ]

    if filtered.empty:
        print("Nenhum modelo satisfaz os critérios.")
        return None

    # 🔹 melhor para cada métrica
    best_tr1 = filtered.sort_values(r2_tr1, ascending=False).iloc[0]
    best_tr2 = filtered.sort_values(r2_tr2, ascending=False).iloc[0]
    best_val = filtered.sort_values(r2_val, ascending=False).iloc[0]
    best_t3 = filtered.sort_values(r2_t3, ascending=False).iloc[0]

    selected = pd.DataFrame([best_tr1, best_tr2, best_val, best_t3]).drop_duplicates()

    cols = ["model", "Neurons", "r",  r2_tr1, r2_tr2, r2_val, r2_t3, "Wf"]

    table = selected[cols].copy()

    # 🔹 calcular média das métricas
    mean_values = table[[r2_tr1, r2_tr2, r2_val, r2_t3]].mean()

    mean_row = pd.DataFrame([{
        "model": "MEAN",
        "Neurons": "-",
        "r": "-",
        "Wf": "-",
        r2_tr1: mean_values[r2_tr1],
        r2_tr2: mean_values[r2_tr2],
        r2_val: mean_values[r2_val],
        r2_t3: mean_values[r2_t3]
    }])

    table_mean = pd.concat([table, mean_row], ignore_index=True)

    display(table_mean)

    return table

In [23]:
models = []
models.append(PickModels(results[0], "Theta", tr1=0.8, tr2=0.6, v1=0.6, t3=0.6))
models.append(PickModels(results[1], "X", tr1=0.0, tr2=0.0, v1=0.0, t3=0.0))
models.append(PickModels(results[2], "Y", tr1=0.8, tr2=0.7, v1=0.7, t3=0.7))

,model,Neurons,r,R2_Train_1_Theta,R2_Train_2_Theta,R2_Val_Theta,R2_Test_3_Theta,Wf
0,model_264_128_Ld0.3_Lp0.7_r0.5_seed0,"[264, 128]",0.5,0.882420,0.607998,0.659482,0.758187,"[[[-0.03269999846816063, -0.06109999865293503,..."
1,model_128_64_Ld0.3_Lp0.7_r0.9_seed3,"[128, 64]",0.9,0.846928,0.685326,0.640159,0.713271,"[[[0.09939999878406525, 0.08860000222921371, -..."
2,MEAN,-,-,0.864674,0.646662,0.649820,0.735729,-


,model,Neurons,r,R2_Train_1_X,R2_Train_2_X,R2_Val_X,R2_Test_3_X,Wf
0,model_32_1,[32],0.01,0.824865,0.944005,0.884637,0.074087,"[[[-0.07660000026226044, 0.06679999828338623, ..."
1,model_4_3,"[8, 4]",0.01,0.454366,0.970695,0.896838,0.338232,"[[[-0.009399999864399433, 0.03229999914765358,..."
2,MEAN,-,-,0.639615,0.957350,0.890738,0.206159,-


,model,Neurons,r,R2_Train_1_Y,R2_Train_2_Y,R2_Val_Y,R2_Test_3_Y,Wf
0,model_264_4,[264],0.5,0.803500,0.792559,0.857673,0.872257,"[[[-0.0010000000474974513, -0.0003000000142492..."
1,model_32_2,"[64, 32]",0.5,0.802206,0.904358,0.759207,0.790826,"[[[-0.00039999998989515007, -0.006300000008195..."
2,MEAN,-,-,0.802853,0.848458,0.808440,0.831542,-


In [24]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

def CreateSequences(input_data, target_data, timesteps):

    X_seq, Y_seq = [], []

    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i].values)

    return np.array(X_seq), np.array(Y_seq)


class EnsembleNets():

    def __init__(self, predictors, targets, int_targets, out_scaler):

        self.models = []
        self.predictors = predictors
        self.targets = targets
        self.target_int = int_targets
        self.input_size = len(predictors)
        self.output_size = len(targets)
        self.out_scaler = out_scaler


    def SetModels(self, architecture, Wf, r):

        regularizer = tf.keras.regularizers.l2(r)

        model = tf.keras.Sequential()
        model.add(tf.keras.layers.Input(shape=(TIME_STEPS, self.input_size)))

        for i, units in enumerate(architecture):

            if i < len(architecture) - 1:
                model.add(
                    tf.keras.layers.SimpleRNN(
                        units,
                        activation="tanh",
                        return_sequences=True,
                        kernel_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )

            else:
                model.add(
                    tf.keras.layers.SimpleRNN(
                        units,
                        activation="tanh",
                        kernel_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )

        model.add(
            tf.keras.layers.Dense(
                self.output_size,
                activation="linear",
                kernel_regularizer=regularizer,
                bias_regularizer=regularizer
            )
        )

        model.set_weights(Wf)
        self.models.append(model)


    def BuildEnsemble(self, ModelsList):
        for (arch, Wf, r) in ModelsList:
            if isinstance(arch, str):
                arch = ast.literal_eval(arch)
            Wf = eval(Wf)
            Wf = [np.array(w) for w in Wf]   
            self.SetModels(arch, Wf, r)


    def PredictModel(self, model, x):

        y_pred = model.predict(x)
        y_pred = self.out_scaler.inverse_transform(y_pred)
        init_vals = np.array([Datasets[i][name].iloc[0] for name in self.target_int])
        
        for j in range(self.output_size):
            y_pred[:, j] = init_vals[j] + np.cumsum(y_pred[:, j] * TS)
        return y_pred


    def EvalEnsemble(self, datasets, Wensemble):

        for d, dataset in enumerate(datasets):

            x, y = CreateSequences(
                dataset[self.predictors],
                dataset[self.targets],
                TIME_STEPS
            )

            y_true = dataset[self.target_int].iloc[TIME_STEPS:].values

            y_ensemble = np.zeros((len(x), self.output_size))

            print(f"\n===== DATASET: {SETS[d]} =====")

            # 🔹 métricas individuais
            for i, model in enumerate(self.models):

                y_pred = self.PredictModel(model, x)

                y_ensemble += Wensemble[i] * y_pred

                for j, name in enumerate(self.target_int):

                    r2 = r2_score(y_true[:, j], y_pred[:, j])
                    mse = mean_squared_error(y_true[:, j], y_pred[:, j])

                    print(
                        f"Model {i+1} | {name} -> "
                        f"R² = {r2:.4f}, MSE = {mse:.4e}"
                    )

            # 🔹 métricas do ensemble
            print("---- ENSEMBLE ----")

            for j, name in enumerate(self.target_int):

                r2 = r2_score(y_true[:, j], y_ensemble[:, j])
                mse = mean_squared_error(y_true[:, j], y_ensemble[:, j])

                print(
                    f"Ensemble | {name} -> "
                    f"R² = {r2:.4f}, MSE = {mse:.4e}"
                )

In [25]:
def Test(predictors, targets, int_targets, out_scaler, i):
    ModelsList = []

    for _, row in models[i].iterrows():

        arch = row["Neurons"]
        r = float(row["r"])
        Wf = row["Wf"]

        ModelsList.append((arch, Wf, r))
        
    Wensemble = np.ones(len(ModelsList)) / len(ModelsList) 
    
    ensemble = EnsembleNets(
        predictors=predictors,
        targets=targets,
        int_targets=int_targets,
        out_scaler=out_scaler,
    )

    ensemble.BuildEnsemble(ModelsList)
    ensemble.EvalEnsemble(
        datasets=NormDatasets,
        Wensemble=Wensemble,
    )

In [26]:
Test(predictors=["PwmD", "PwmE", "Theta"],
     targets=["DeltaX"],
     int_targets=["X"],
     out_scaler=OUT_SCALERS["DeltaX"], 
     i=1)


===== DATASET: Train_1 =====
31/31 [==============================] - 0s 2ms/step
Model 1 | X -> R² = 0.8261, MSE = 1.8758e-02
31/31 [==============================] - 0s 2ms/step
Model 2 | X -> R² = 0.4585, MSE = 5.8392e-02
---- ENSEMBLE ----
Ensemble | X -> R² = 0.6708, MSE = 3.5500e-02

===== DATASET: Train_2 =====
31/31 [==============================] - 0s 1ms/step
Model 1 | X -> R² = 0.9507, MSE = 4.1557e-03
31/31 [==============================] - 0s 1ms/step
Model 2 | X -> R² = 0.9594, MSE = 3.4251e-03
---- ENSEMBLE ----
Ensemble | X -> R² = 0.9789, MSE = 1.7819e-03

===== DATASET: Test_1 =====
31/31 [==============================] - 0s 1ms/step
Model 1 | X -> R² = 0.8812, MSE = 1.2445e-02
31/31 [==============================] - 0s 1ms/step
Model 2 | X -> R² = 0.6110, MSE = 4.0759e-02
---- ENSEMBLE ----
Ensemble | X -> R² = 0.7678, MSE = 2.4326e-02

===== DATASET: Test_2 =====
31/31 [==============================] - 0s 1ms/step
Model 1 | X -> R² = 0.9566, MSE = 3.5704e-03
3

In [27]:
Test(predictors=["PwmD", "PwmE", "Theta"],
     targets=["DeltaY"],
     int_targets=["Y"],
     out_scaler=OUT_SCALERS["DeltaY"], 
     i=2)

SyntaxError: '[' was never closed (<string>, line 1)